<a href="https://colab.research.google.com/github/g20264006-glitch/ADSASDSASDASADSADASDASDDADAADSDAASDDSADSADAA/blob/main/02_ai_dashboard_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2차시: AI로 대여량 예측하고 웹 대시보드 만들기
## 학생용 따라하기 · 180분

**오늘의 중심 질문**

> 시간과 날씨 정보를 사용하면 AI가 자전거 대여량을 어느 정도 예측할 수 있을까?

**완성 결과**

1. 평균만 사용하는 기준 예측
2. 의사결정나무 AI 예측
3. 실제값과 예측값 비교 그래프
4. AI가 크게 틀린 사례 분석
5. GitHub Pages에 올릴 `docs/index.html` 대시보드

오늘의 핵심은 높은 점수를 만드는 것이 아니라, **AI를 기준 방법과 비교하고 오류를 설명하는 것**입니다.

## 수업 흐름

| 시간 | 활동 | 중간 산출물 |
|---:|---|---|
| 0~20분 | 1차시 복습과 예측 문제 정의 | 입력·정답 구분 |
| 20~50분 | 과거·미래 순서로 학습/시험 분리 | 학습용·시험용 표 |
| 50~85분 | 기준 예측과 AI 모델 비교 | MAE 결과표 |
| 85~95분 | 휴식 |  |
| 95~125분 | 예측 그래프·오류·중요 변수 분석 | AI 분석 결과 |
| 125~160분 | HTML 대시보드 자동 생성 | `docs/index.html` |
| 160~180분 | GitHub 업로드·Pages 설정·성찰 | 공개 링크 또는 제출 파일 |

## 0. AI 예측 문제 이해하기

AI가 사용할 정보인 **입력 변수**와 맞히려는 값인 **예측 대상**을 구분합니다.

| 역할 | 이 수업의 예 |
|---|---|
| 입력 변수(X) | 시간, 기온, 습도, 강수량, 풍속, 주말, 휴일 |
| 예측 대상(y) | 한 시간 동안의 자전거 대여량 |

AI는 모든 원인을 아는 것이 아니라, 과거 표에서 입력과 대여량 사이의 규칙을 찾습니다.

## 1. 라이브러리와 데이터 준비 · 그대로 실행

1차시와 같은 UCI 원본을 다시 불러옵니다. 따라서 이 노트북만 따로 열어도 실행됩니다.

In [1]:
# 데이터 불러오기: UCI 공식 압축 파일을 먼저 사용하고,
# 접속이 어려우면 검증용 GitHub 미러를 사용합니다.
import io
import zipfile
from pathlib import Path

import pandas as pd
import requests

OFFICIAL_ZIP_URL = "https://archive.ics.uci.edu/static/public/560/seoul%2Bbike%2Bsharing%2Bdemand.zip"
MIRROR_CSV_URL = "https://raw.githubusercontent.com/PranavUikey/DS_Extra/main/Datasets%20for%20EDA/seoul_bicycle_dataset/SeoulBikeData.csv"


def load_seoul_bike_data():
    try:
        response = requests.get(OFFICIAL_ZIP_URL, timeout=30)
        response.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
            csv_names = [
                name for name in archive.namelist()
                if name.lower().endswith(".csv")
            ]
            if not csv_names:
                raise FileNotFoundError("압축 파일 안에서 CSV를 찾지 못했습니다.")
            with archive.open(csv_names[0]) as csv_file:
                data = pd.read_csv(csv_file, encoding="unicode_escape")
        return data, "UCI 공식 저장소"
    except Exception as official_error:
        print("UCI 직접 연결 실패. 공개 미러로 다시 시도합니다.")
        print("오류 요약:", type(official_error).__name__)
        data = pd.read_csv(MIRROR_CSV_URL, encoding="utf-8")
        return data, "GitHub 공개 미러(UCI 원본 복제본)"


raw_df, data_source_used = load_seoul_bike_data()
print("사용한 데이터 경로:", data_source_used)
print("원본 크기:", raw_df.shape)

사용한 데이터 경로: UCI 공식 저장소
원본 크기: (8760, 14)


In [2]:
# 열 이름을 학생이 읽기 쉬운 한글로 바꿉니다.
KOREAN_COLUMNS = [
    "날짜", "대여량", "시간", "기온", "습도", "풍속", "가시거리",
    "이슬점", "일사량", "강수량", "적설량", "계절", "휴일", "운영여부"
]

if raw_df.shape[1] != len(KOREAN_COLUMNS):
    raise ValueError(
        f"예상한 열은 {len(KOREAN_COLUMNS)}개인데, "
        f"실제 데이터에는 {raw_df.shape[1]}개가 있습니다."
    )

df = raw_df.copy()
df.columns = KOREAN_COLUMNS

# 날짜를 컴퓨터가 이해하는 날짜 자료형으로 바꿉니다.
df["날짜"] = pd.to_datetime(df["날짜"], format="%d/%m/%Y")

# 자전거 시스템이 정상 운영된 시간만 분석합니다.
df = df[df["운영여부"] == "Yes"].copy()

# 분석에 도움이 되는 새 열을 만듭니다.
요일_이름 = {0: "월", 1: "화", 2: "수", 3: "목", 4: "금", 5: "토", 6: "일"}
df["요일"] = df["날짜"].dt.dayofweek.map(요일_이름)
df["주말"] = (df["날짜"].dt.dayofweek >= 5).astype(int)
df["주말여부"] = df["주말"].map({0: "평일", 1: "주말"})
df["휴일여부"] = (df["휴일"] == "Holiday").astype(int)
df["비여부"] = (df["강수량"] > 0).map({False: "비 없음", True: "비 있음"})
df["날짜시간"] = df["날짜"] + pd.to_timedelta(df["시간"], unit="h")

df = df.sort_values(["날짜", "시간"]).reset_index(drop=True)
print("분석에 사용할 크기:", df.shape)

분석에 사용할 크기: (8465, 20)


In [3]:
import plotly.express as px
import plotly.io as pio

# 그래프의 기본 모양을 통일합니다.
pio.templates.default = "plotly_white"

In [17]:
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.tree import DecisionTreeRegressor

## 2. AI가 사용할 표 만들기 · 읽고 실행

이번 모델에서는 이해하기 쉬운 숫자 정보만 사용합니다.
날짜 자체는 입력하지 않지만, 날짜에서 만든 `주말`과 `휴일여부`는 사용합니다.

In [18]:
ALL_FEATURES = [
    "시간", "기온", "습도", "풍속", "강수량", "주말", "휴일여부"
]
TARGET = "대여량"

model_df = (
    df[["날짜시간", TARGET] + ALL_FEATURES]
    .dropna()
    .sort_values("날짜시간")
    .reset_index(drop=True)
)

display(model_df.head())
print("모델용 기록 수:", len(model_df))

,날짜시간,대여량,시간,기온,습도,풍속,강수량,주말,휴일여부
0,2017-12-01 00:00:00,254,0,-5.2,37,2.2,0.0,0,0
1,2017-12-01 01:00:00,204,1,-5.5,38,0.8,0.0,0,0
2,2017-12-01 02:00:00,173,2,-6.0,39,1.0,0.0,0,0
3,2017-12-01 03:00:00,107,3,-6.2,40,0.9,0.0,0,0
4,2017-12-01 04:00:00,78,4,-6.0,36,2.3,0.0,0,0


모델용 기록 수: 8465


## 3. 과거는 학습, 뒤의 기간은 시험 · 읽고 실행

미래를 예측하는 상황을 흉내 내기 위해 시간 순서를 유지합니다.

- 앞의 80%: AI가 규칙을 배우는 **학습 데이터**
- 뒤의 20%: 학습에 사용하지 않고 실력을 확인하는 **시험 데이터**

시험 데이터의 정답을 학습 과정에 미리 보여 주면 공정한 평가가 아닙니다.

In [19]:
split_index = int(len(model_df) * 0.8)
train_df = model_df.iloc[:split_index].copy()
test_df = model_df.iloc[split_index:].copy()

print("학습 데이터:", len(train_df), "개")
print("시험 데이터:", len(test_df), "개")
print("학습 기간:", train_df["날짜시간"].min(), "~", train_df["날짜시간"].max())
print("시험 기간:", test_df["날짜시간"].min(), "~", test_df["날짜시간"].max())

학습 데이터: 6772 개
시험 데이터: 1693 개
학습 기간: 2017-12-01 00:00:00 ~ 2018-09-11 03:00:00
시험 기간: 2018-09-11 04:00:00 ~ 2018-11-30 23:00:00


## 4. 먼저 아주 단순한 기준 방법 만들기 · 읽고 실행

AI가 유용하다고 말하려면 비교 대상이 필요합니다. 가장 단순한 기준은
**학습 기간의 평균 대여량을 모든 시간에 똑같이 예측하는 방법**입니다.

**MAE(평균절대오차)**는 예측이 실제값에서 평균적으로 몇 대 정도 벗어났는지를 뜻합니다.
숫자가 작을수록 실제값에 더 가깝습니다.

In [20]:
baseline_value = train_df[TARGET].mean()
baseline_prediction = np.full(len(test_df), baseline_value)
baseline_mae = mean_absolute_error(test_df[TARGET], baseline_prediction)

print(f"기준 예측값: 매시간 {baseline_value:,.1f}대")
print(f"기준 방법 MAE: {baseline_mae:,.1f}대")

기준 예측값: 매시간 687.5대
기준 방법 MAE: 491.5대


## 5. 의사결정나무 AI 학습 · 읽고 실행

의사결정나무는 “시간이 7시보다 이른가?”, “비가 왔는가?”와 같은 질문을 반복하며
데이터를 나누고 예측 규칙을 만듭니다.

`max_depth`가 너무 크면 학습 데이터만 지나치게 외울 수 있으므로 깊이를 제한합니다.

In [21]:
X_train = train_df[ALL_FEATURES]
y_train = train_df[TARGET]
X_test = test_df[ALL_FEATURES]
y_test = test_df[TARGET]

model = DecisionTreeRegressor(
    max_depth=8,
    min_samples_leaf=15,
    random_state=42
)
model.fit(X_train, y_train)
ai_prediction = model.predict(X_test)
ai_mae = mean_absolute_error(y_test, ai_prediction)

print(f"기준 방법 MAE: {baseline_mae:,.1f}대")
print(f"AI 모델 MAE: {ai_mae:,.1f}대")

기준 방법 MAE: 491.5대
AI 모델 MAE: 279.7대


## 6. 기준 방법과 AI 비교 · 읽고 실행

In [22]:
comparison = pd.DataFrame({
    "방법": ["평균만 사용", "의사결정나무 AI"],
    "MAE_대": [baseline_mae, ai_mae]
})
comparison["MAE_대"] = comparison["MAE_대"].round(1)
comparison["기준보다_개선율_%"] = [
    0,
    round((baseline_mae - ai_mae) / baseline_mae * 100, 1)
]
comparison

,방법,MAE_대,기준보다_개선율_%
0,평균만 사용,491.5,0.0
1,의사결정나무 AI,279.7,43.1


In [24]:
if ai_mae < baseline_mae:
    print(
        f"AI가 평균 기준보다 평균 오차를 "
        f"{baseline_mae - ai_mae:,.1f}대 줄였습니다."
    )
else:
    print(
        "이번 설정에서는 AI가 평균 기준보다 낫지 않았습니다. "
        "이 결과도 중요한 분석 결과입니다."
    )

AI가 평균 기준보다 평균 오차를 211.8대 줄였습니다.


## 7. 세 가지 입력 조합 실험 · 읽고 실행

같은 알고리즘에 어떤 정보를 주느냐에 따라 결과가 달라지는지 확인합니다.

- A: 시간 정보만
- B: 날씨 정보만
- C: 시간과 날씨를 함께

In [25]:
FEATURE_EXPERIMENTS = {
    "A_시간정보": ["시간", "주말", "휴일여부"],
    "B_날씨정보": ["기온", "습도", "풍속", "강수량"],
    "C_시간과날씨": ALL_FEATURES
}

experiment_rows = []
experiment_models = {}

for experiment_name, features in FEATURE_EXPERIMENTS.items():
    experiment_model = DecisionTreeRegressor(
        max_depth=8,
        min_samples_leaf=15,
        random_state=42
    )
    experiment_model.fit(train_df[features], y_train)
    prediction = experiment_model.predict(test_df[features])
    mae = mean_absolute_error(y_test, prediction)

    experiment_rows.append({
        "실험": experiment_name,
        "사용한_정보": ", ".join(features),
        "MAE_대": round(mae, 1)
    })
    experiment_models[experiment_name] = (
        experiment_model,
        features,
        prediction
    )

experiment_result = (
    pd.DataFrame(experiment_rows)
    .sort_values("MAE_대")
    .reset_index(drop=True)
)
experiment_result

,실험,사용한_정보,MAE_대
0,C_시간과날씨,"시간, 기온, 습도, 풍속, 강수량, 주말, 휴일여부",279.7
1,A_시간정보,"시간, 주말, 휴일여부",327.9
2,B_날씨정보,"기온, 습도, 풍속, 강수량",393.2


### 실험 해석 질문 · 직접 작성

1. 어떤 입력 조합의 MAE가 가장 작았나요?
2. 시간 정보만 쓴 결과와 날씨 정보만 쓴 결과는 어떻게 달랐나요?
3. 정보를 더 많이 주면 항상 성능이 좋아졌나요?
4. 이 데이터에 어떤 정보를 더 추가하면 좋을까요?

## 8. 실제값과 AI 예측값 비교 · 읽고 실행

가장 많은 정보를 사용한 C 실험의 결과를 시간 순서로 비교합니다.
그래프가 너무 빽빽하지 않도록 시험 기간의 마지막 7일만 보여 줍니다.

In [26]:
result = test_df[["날짜시간", TARGET]].copy()
result["AI예측"] = ai_prediction
result["절대오차"] = (result[TARGET] - result["AI예측"]).abs()

last_week = result.tail(24 * 7).copy()
prediction_long = last_week.melt(
    id_vars="날짜시간",
    value_vars=[TARGET, "AI예측"],
    var_name="구분",
    value_name="대여량값"
)

fig_prediction = px.line(
    prediction_long,
    x="날짜시간",
    y="대여량값",
    color="구분",
    title="시험 기간 마지막 7일: 실제 대여량과 AI 예측",
    labels={"날짜시간": "날짜와 시간", "대여량값": "대여량(대)", "구분": "값"}
)
fig_prediction.show()

## 9. AI가 크게 틀린 사례 찾기 · 읽고 실행

좋은 모델도 항상 맞지는 않습니다. 오류가 큰 행을 찾아 입력 변수만으로 설명되지 않는
특별한 상황이 있었을 가능성을 생각합니다.

In [27]:
error_cases = (
    result.nlargest(10, "절대오차")
    .merge(
        model_df[["날짜시간"] + ALL_FEATURES],
        on="날짜시간",
        how="left"
    )
)
error_cases[[
    "날짜시간", "대여량", "AI예측", "절대오차",
    "시간", "기온", "습도", "강수량", "주말", "휴일여부"
]].round(1)

,날짜시간,대여량,AI예측,절대오차,시간,기온,습도,강수량,주말,휴일여부
0,2018-10-11 18:00:00,2378,640.7,1737.3,18,12.1,40,0.0,0,0
1,2018-10-18 08:00:00,2154,539.2,1614.8,8,8.5,77,0.0,0,0
2,2018-11-01 18:00:00,2254,640.7,1613.3,18,11.8,43,0.0,0,0
3,2018-10-19 08:00:00,2113,539.2,1573.8,8,7.9,63,0.0,0,0
4,2018-10-24 08:00:00,2108,539.2,1568.8,8,8.1,85,0.0,0,0
5,2018-10-25 08:00:00,2070,539.2,1530.8,8,7.9,76,0.0,0,0
6,2018-11-13 18:00:00,2159,640.7,1518.3,18,11.3,42,0.0,0,0
7,2018-09-14 22:00:00,567,2074.3,1507.3,22,24.0,73,0.0,0,0
8,2018-11-16 18:00:00,1956,468.3,1487.7,18,9.9,74,0.0,0,0
9,2018-10-12 08:00:00,1996,539.2,1456.8,8,6.0,64,0.0,0,0


### 오류 사례 토의

큰 오류가 생긴 이유로 가능한 것을 고릅니다.

- 데이터에 없는 지역 행사나 집회
- 지하철·버스 운행 변화
- 대여소별 자전거 부족
- 갑작스러운 날씨 변화
- 학교 방학이나 특별 휴무
- 의사결정나무의 단순한 규칙

표만으로 원인을 확정하지 말고 **가능성**으로 표현합니다.

## 10. AI가 중요하게 사용한 정보 · 읽고 실행

`feature_importances_`는 이번 의사결정나무가 데이터를 나눌 때 각 정보를
얼마나 많이 활용했는지 보여 줍니다. 이것은 원인의 크기가 아니라 **모델 안에서의 활용도**입니다.

In [28]:
importance = pd.DataFrame({
    "입력변수": ALL_FEATURES,
    "중요도": model.feature_importances_
}).sort_values("중요도", ascending=False)

fig_importance = px.bar(
    importance,
    x="중요도",
    y="입력변수",
    orientation="h",
    title="AI가 예측할 때 많이 사용한 정보",
    labels={"입력변수": "입력 정보", "중요도": "모델 중요도"}
)
fig_importance.update_layout(yaxis={"categoryorder": "total ascending"})
fig_importance.show()

## 11. 대시보드용 분석 그래프 준비 · 그대로 실행

In [29]:
hourly_mean = (
    df.groupby("시간", as_index=False)["대여량"]
      .mean()
      .rename(columns={"대여량": "평균대여량"})
)
rain_mean = (
    df.groupby("비여부", as_index=False)["대여량"]
      .mean()
      .rename(columns={"대여량": "평균대여량"})
)
daily = (
    df.groupby("날짜", as_index=False)
      .agg(
          총대여량=("대여량", "sum"),
          평균기온=("기온", "mean"),
          총강수량=("강수량", "sum")
      )
)

fig_hour = px.bar(
    hourly_mean,
    x="시간",
    y="평균대여량",
    title="시간대별 평균 대여량",
    labels={"시간": "시간(시)", "평균대여량": "평균 대여량(대)"}
)
fig_hour.update_layout(xaxis=dict(dtick=1))

fig_rain = px.bar(
    rain_mean,
    x="비여부",
    y="평균대여량",
    title="강수 여부에 따른 평균 대여량",
    labels={"비여부": "강수 여부", "평균대여량": "평균 대여량(대)"},
    text_auto=".0f"
)

fig_daily = px.line(
    daily,
    x="날짜",
    y="총대여량",
    title="날짜별 하루 총대여량",
    labels={"날짜": "날짜", "총대여량": "총대여량(대)"}
)

## 12. 대시보드에 넣을 설명 작성 · 직접 수정

아래 네 문장을 자신의 분석 결과에 맞게 바꿉니다. 문장에는 가능하면 숫자 근거를 넣습니다.

In [35]:
PROJECT_TITLE = "서울 공공자전거: 날씨와 시간에 따른 대여량 분석"
MAIN_QUESTION = "시간과 날씨 정보로 자전거 대여량을 설명하고 예측할 수 있을까?"

FINDINGS = [
    "[우리의 발견]",
    "1. 시간대별 그래프에서 알아보기 편했다",
    "2. 강수 여부를 비교했을 때 신기했다",
    "3. 기온과 대여량의 산점도에서 신기했다",
    "[분석의 한계]",
    "하지만 이 데이터에는 실시간 정보가 없으므로 확실히 단정할 수 없다."
]

LIMITATIONS = [
    "이 결과는 2017~2018년의 서울 공공자전거 기록에 기반한다.",
    "행사, 대여소 위치, 자전거 재배치 등 중요한 정보가 포함되지 않았다.",
    "날씨와 대여량의 관계가 보이더라도 날씨가 유일한 원인이라고 단정할 수 없다.",
    "AI 예측은 수업용 실험이며 실제 운영 결정을 대신할 수 없다."
]

## 13. HTML 대시보드 자동 생성 · 그대로 실행

HTML과 CSS를 직접 학습하는 시간이 아니라 데이터 분석 수업이므로,
아래 함수가 그래프와 설명을 하나의 웹페이지로 자동 조립합니다.

Plotly 그래프는 HTML 안에서도 확대, 범례 선택, 마우스 오버가 가능합니다.

In [36]:
from html import escape
from pathlib import Path

DATASET_PAGE = "https://archive.ics.uci.edu/dataset/560/seoul%2Bbike%2Bsharing%2Bdemand"
DATASET_DOI = "https://doi.org/10.24432/C5F62R"


def figure_html(figure, include_js=False):
    return figure.to_html(
        full_html=False,
        include_plotlyjs="cdn" if include_js else False,
        config={
            "displaylogo": False,
            "responsive": True
        }
    )


def build_dashboard(
    title,
    question,
    figures,
    findings,
    limitations,
    output_path="docs/index.html"
):
    peak_hour = int(
        hourly_mean.loc[hourly_mean["평균대여량"].idxmax(), "시간"]
    )
    best_experiment_name = experiment_result.iloc[0]["실험"]

    finding_items = "".join(
        f"<li>{escape(str(item))}</li>" for item in findings
    )
    limitation_items = "".join(
        f"<li>{escape(str(item))}</li>" for item in limitations
    )

    chart_sections = []
    for index, (heading, figure, explanation) in enumerate(figures):
        chart_sections.append(f"""
        <section class="panel chart-panel">
            <h2>{escape(heading)}</h2>
            <p class="chart-note">{escape(explanation)}</p>
            {figure_html(figure, include_js=(index == 0))}
        </section>
        """)

    html = f"""<!doctype html>
<html lang="ko">
<head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title>{escape(title)}</title>
    <style>
        :root {{
            --ink: #172033;
            --muted: #5f6b7a;
            --surface: #ffffff;
            --soft: #f3f6fb;
            --line: #dfe5ee;
            --accent: #2457d6;
        }}
        * {{ box-sizing: border-box; }}
        body {{
            margin: 0;
            background: var(--soft);
            color: var(--ink);
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI",
                         "Noto Sans KR", Arial, sans-serif;
            line-height: 1.65;
        }}
        .wrap {{ max-width: 1180px; margin: 0 auto; padding: 28px 18px 64px; }}
        .hero {{
            background: linear-gradient(135deg, #172033, #2457d6);
            color: white;
            padding: 38px;
            border-radius: 22px;
            box-shadow: 0 16px 40px rgba(23, 32, 51, .16);
        }}
        .eyebrow {{ margin: 0 0 8px; font-weight: 700; opacity: .82; }}
        h1 {{ margin: 0; font-size: clamp(1.8rem, 4vw, 3rem); line-height: 1.22; }}
        .question {{ margin: 16px 0 0; font-size: 1.08rem; max-width: 850px; }}
        .kpis {{
            display: grid;
            grid-template-columns: repeat(4, minmax(0, 1fr));
            gap: 14px;
            margin: 20px 0;
        }}
        .kpi, .panel {{
            background: var(--surface);
            border: 1px solid var(--line);
            border-radius: 18px;
            box-shadow: 0 8px 24px rgba(23, 32, 51, .06);
        }}
        .kpi {{ padding: 20px; }}
        .kpi-label {{ color: var(--muted); font-size: .9rem; }}
        .kpi-value {{ margin-top: 4px; font-size: 1.55rem; font-weight: 800; }}
        .panel {{ padding: 24px; margin-top: 18px; }}
        .panel h2 {{ margin: 0 0 8px; font-size: 1.35rem; }}
        .chart-note {{ color: var(--muted); margin: 0 0 12px; }}
        .two-column {{
            display: grid;
            grid-template-columns: repeat(2, minmax(0, 1fr));
            gap: 18px;
        }}
        li {{ margin: 7px 0; }}
        .source {{ color: var(--muted); font-size: .92rem; }}
        a {{ color: var(--accent); }}
        footer {{ margin-top: 20px; text-align: center; color: var(--muted); }}
        @media (max-width: 800px) {{
            .kpis, .two-column {{ grid-template-columns: 1fr; }}
            .hero {{ padding: 26px; }}
            .panel {{ padding: 16px; }}
        }}
    </style>
</head>
<body>
    <main class="wrap">
        <header class="hero">
            <p class="eyebrow">AI 데이터 탐정 프로젝트</p>
            <h1>{escape(title)}</h1>
            <p class="question"><strong>중심 질문:</strong> {escape(question)}</p>
        </header>

        <section class="kpis" aria-label="핵심 지표">
            <article class="kpi">
                <div class="kpi-label">분석 기록</div>
                <div class="kpi-value">{len(df):,}개</div>
            </article>
            <article class="kpi">
                <div class="kpi-label">평균 시간당 대여량</div>
                <div class="kpi-value">{df['대여량'].mean():,.0f}대</div>
            </article>
            <article class="kpi">
                <div class="kpi-label">평균 대여량 최고 시간</div>
                <div class="kpi-value">{peak_hour}시</div>
            </article>
            <article class="kpi">
                <div class="kpi-label">AI 평균절대오차</div>
                <div class="kpi-value">{ai_mae:,.0f}대</div>
            </article>
        </section>

        {''.join(chart_sections)}

        <section class="two-column">
            <article class="panel">
                <h2>데이터에서 발견한 것</h2>
                <ol>{finding_items}</ol>
            </article>
            <article class="panel">
                <h2>해석할 때 주의할 점</h2>
                <ul>{limitation_items}</ul>
            </article>
        </section>

        <section class="panel">
            <h2>AI 실험 요약</h2>
            <p>
                평균만 사용하는 기준의 MAE는 <strong>{baseline_mae:,.1f}대</strong>,
                시간·날씨 정보를 함께 사용한 의사결정나무의 MAE는
                <strong>{ai_mae:,.1f}대</strong>였습니다.
                세 입력 조합 중 가장 작은 MAE를 보인 실험은
                <strong>{escape(str(best_experiment_name))}</strong>입니다.
            </p>
            <p class="source">
                MAE는 시험 자료에서 예측이 실제값과 평균적으로 얼마나 벗어났는지를 나타냅니다.
                낮을수록 가깝지만, 이 한 지표만으로 실제 사용 가능성을 판단할 수는 없습니다.
            </p>
        </section>

        <section class="panel source">
            <h2>데이터 출처와 재현 정보</h2>
            <p>
                Seoul Bike Sharing Demand, UCI Machine Learning Repository,
                DOI: <a href="{DATASET_DOI}">10.24432/C5F62R</a>, CC BY 4.0.
            </p>
            <p>
                분석은 Python, pandas, Plotly, scikit-learn을 사용했습니다.
                전체 계산 과정은 같은 GitHub 저장소의 Colab/Jupyter 노트북에 기록합니다.
            </p>
        </section>
        <footer>수업용 데이터 분석 대시보드</footer>
    </main>
</body>
</html>"""

    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(html, encoding="utf-8")
    (output.parent / ".nojekyll").write_text("", encoding="utf-8")
    return output


dashboard_figures = [
    (
        "1. 시간대별 평균 대여량",
        fig_hour,
        "하루 24시간의 평균 대여량을 비교합니다."
    ),
    (
        "2. 날짜별 하루 총대여량",
        fig_daily,
        "한 해 동안 대여량이 어떻게 변했는지 확인합니다."
    ),
    (
        "3. 강수 여부에 따른 평균 대여량",
        fig_rain,
        "비가 기록된 시간과 그렇지 않은 시간을 비교합니다."
    ),
    (
        "4. 실제값과 AI 예측값",
        fig_prediction,
        "시험 기간 마지막 7일의 실제값과 예측값을 시간 순서로 비교합니다."
    ),
    (
        "5. AI가 활용한 입력 정보",
        fig_importance,
        "의사결정나무 안에서 각 입력 변수가 얼마나 활용되었는지 보여 줍니다."
    )
]

dashboard_path = build_dashboard(
    title=PROJECT_TITLE,
    question=MAIN_QUESTION,
    figures=dashboard_figures,
    findings=FINDINGS,
    limitations=LIMITATIONS
)

print("대시보드 생성 완료:", dashboard_path.resolve())
print("GitHub에는 docs/index.html과 docs/.nojekyll을 올립니다.")

대시보드 생성 완료: /content/docs/index.html
GitHub에는 docs/index.html과 docs/.nojekyll을 올립니다.


## 14. Colab에서 대시보드 확인하기 · 그대로 실행

아래 셀은 생성된 HTML을 새 브라우저 파일로 내려받습니다.
파일을 열어 그래프의 마우스 오버·확대·범례 선택이 작동하는지 확인합니다.

In [37]:
from IPython.display import display, HTML

print("생성된 파일:", dashboard_path)
display(HTML(
    '<p><strong>확인:</strong> 왼쪽 파일 목록에서 '
    '<code>docs/index.html</code>을 찾을 수 있습니다.</p>'
))

생성된 파일: docs/index.html


In [38]:
# Colab에서 내려받으려면 이 셀의 주석을 해제하고 실행합니다.
# from google.colab import files
# files.download("docs/index.html")

## 15. Colab 결과를 GitHub로 옮기기

### A. 분석 노트북 저장

1. Colab 메뉴에서 **파일 → GitHub에 사본 저장**을 선택합니다.
2. 자신의 저장소를 선택합니다.
3. 파일 경로를 `notebooks/02_ai_dashboard.ipynb`로 정합니다.
4. 저장소에서 노트북이 열리는지 확인합니다.

### B. 대시보드 파일 업로드

1. Colab 왼쪽의 폴더 아이콘에서 `docs/index.html`을 내려받습니다.
2. GitHub 저장소에서 `docs` 폴더를 엽니다.
3. **Add file → Upload files**로 `index.html`을 올립니다.
4. 가능하면 빈 파일 `.nojekyll`도 함께 올립니다. 없어도 이 단일 HTML은 대체로 작동합니다.
5. 커밋 메시지를 `Add AI data dashboard`로 입력합니다.

### C. GitHub Pages 켜기

1. 저장소의 **Settings → Pages**로 이동합니다.
2. **Build and deployment**에서 `Deploy from a branch`를 선택합니다.
3. Branch는 `main`, 폴더는 `/docs`를 선택하고 저장합니다.
4. Pages 배포 작업이 성공한 뒤 표시되는 웹 주소를 엽니다.

저장소 구조는 다음과 같이 정리합니다.

```text
my-data-project/
├── README.md
├── notebooks/
│   ├── 01_data_analysis.ipynb
│   └── 02_ai_dashboard.ipynb
├── data/
│   └── data_source.md
└── docs/
    ├── .nojekyll
    └── index.html
```

## 16. README에 대시보드 연결하기

패키지의 `templates/README_template.md`를 복사한 뒤 자신의 저장소 주소에 맞게 수정합니다.

```markdown
## 결과 보기

- [데이터 분석 대시보드](https://사용자이름.github.io/저장소이름/)
- [분석 노트북](notebooks/02_ai_dashboard.ipynb)
```

GitHub Pages에 학생 실명, 이메일, 개인 설문 원자료, API 키를 올리지 않습니다.

## 17. 2차시 종료 점검

- [ ] 입력 변수와 예측 대상을 구분할 수 있다.
- [ ] 학습 데이터와 시험 데이터를 시간 순서로 나눈 이유를 설명할 수 있다.
- [ ] MAE를 “평균적으로 몇 대 벗어났는가”로 해석할 수 있다.
- [ ] 평균 기준과 AI를 비교했다.
- [ ] AI가 크게 틀린 사례를 확인했다.
- [ ] 변수 중요도를 원인으로 단정하지 않았다.
- [ ] `docs/index.html`을 만들었다.
- [ ] GitHub 저장소와 대시보드 공개 주소를 정리했다.

**마지막 성찰**

1. AI가 잘 맞힌 부분은 무엇인가?
2. AI가 틀린 부분은 무엇인가?
3. 어떤 데이터를 추가하면 더 나아질까?
4. 이 예측을 실제 의사결정에 바로 사용하면 안 되는 이유는 무엇인가?